# 🟢 KaizenStat — Basic Demo (5 min)

**Level:** Beginner | **Time:** ~5 minutes | **Dataset:** Titanic

This notebook teaches the **core 3-step flow**:
1. Load data → `fit()`
2. Check health → `health()`
3. Train a model → `train()`

No prior ML experience needed.

---
> **What you'll learn:**
> - How to get a Data Health Score in one line
> - How KaizenStat auto-selects the best model for you
> - How to read the training output

## Install

In [ ]:
!pip install kaizenstat -q
print("✅ Ready")

## What is KaizenStat?

KaizenStat is a **Python framework for ML pipeline health**. Think of it as a doctor for your dataset:

```
Your data  →  DataDoctor  →  Health Score + Fixed Data + Best Model + Report
```

Instead of writing 200 lines of preprocessing + model selection + debugging code, you write **8 lines**.

## 1. Load the Titanic Dataset

The Titanic dataset is a classic ML dataset. We're predicting whether a passenger **survived** (1) or **died** (0).

It has several real-world messiness issues:
- Missing values in `Age`, `Cabin`, `Embarked`
- Mixed data types (numeric + categorical)
- Some irrelevant columns (`Name`, `Ticket`)

Perfect for showing what KaizenStat does.

In [ ]:
import pandas as pd
from kaizenstat import DataDoctor

# Load Titanic data (public mirror of Kaggle competition data)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Dataset loaded!")
print(f"Shape: {df.shape[0]} passengers, {df.shape[1]} columns")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df.head()

## 2. Register with DataDoctor

`fit()` does three things instantly:
- Validates the dataframe (checks for obvious issues)
- Auto-detects the task type (classification or regression)
- Detects if data is tabular or NLP (text)

No configuration needed.

In [ ]:
doctor = DataDoctor()
doctor.fit(df, target="Survived")

# Check what was detected
print(f"\nMode detected: {doctor.mode()}")
print(f"Doctor state:  {doctor}")

## 3. Data Health Score

This is the first thing you should always check.

The score is 0–100 and covers:
| Check | Penalty if bad |
|-------|---------------|
| Missing values | Up to -30 |
| Duplicate rows | Up to -10 |
| Outliers | Up to -15 |
| Class imbalance | Up to -20 |
| Constant columns | Up to -10 |

**Interpretation:**
- 90–100: Excellent data quality
- 70–89: Minor issues, fix before production
- 50–69: Significant issues, fix before training
- <50: Major problems — training will likely give poor results

In [ ]:
health = doctor.health()

print(f"\n📊 Health Score: {health.score} / 100")

# Show what issues were found
if health.score < 80:
    print("\n⚠️  Issues found — run fix() before training for better results")
else:
    print("\n✅  Data quality is good")

## 4. Quick Fix

Before training, let's clean the data. `fix(safe=True)` applies only **safe, reversible** fixes:
- Fills missing `Age` with median
- Fills missing `Embarked` with mode
- Drops the `Cabin` column (too many missing values)
- Removes duplicate rows (if any)

In [ ]:
fixed_df = doctor.fix(safe=True)

print(f"\nBefore fix: {df.isnull().sum().sum()} missing values")
print(f"After fix:  {fixed_df.isnull().sum().sum()} missing values")
print(f"Rows kept:  {len(fixed_df)} / {len(df)}")

## 5. Train

`train()` automatically:
1. Splits data into train/test (80/20)
2. Benchmarks 5 ML algorithms with cross-validation
3. Selects the winner
4. Trains the final model
5. Reports accuracy, F1, ROC AUC

You don't need to choose an algorithm — KaizenStat picks the best one for your data.

In [ ]:
train_result = doctor.train(cv=5)

print(f"\n🏆 Results:")
print(f"   Best model:  {train_result.model_name}")
print(f"   Test score:  {train_result.test_score:.4f}")
print(f"   Train score: {train_result.train_score:.4f}")

gap = train_result.train_score - train_result.test_score
if gap > 0.1:
    print(f"\n⚠️  Overfitting detected (gap = {gap:.3f}). Consider regularisation.")
elif gap < -0.05:
    print(f"\n⚠️  Underfitting detected (gap = {gap:.3f}). Try a more complex model.")
else:
    print(f"\n✅  Healthy train/test gap ({gap:.3f})")

## 6. Generate Report

Create a shareable HTML report of everything.

In [ ]:
report_path = doctor.report(output_path="basic_titanic_report.html")
print(f"📄 Report saved: {report_path}")

# Display inline
from IPython.display import IFrame, display
display(IFrame(src='basic_titanic_report.html', width='100%', height='500px'))

## 🎯 Try It Yourself

**Exercise 1:** Change the target column
```python
# What if we tried to predict passenger class instead?
doctor2 = DataDoctor()
doctor2.fit(df, target="Pclass")
doctor2.train()
```

**Exercise 2:** Try with your own data
```python
# Upload your CSV to Colab, then:
my_df = pd.read_csv("your_file.csv")
doctor3 = DataDoctor()
doctor3.fit(my_df, target="your_target_column")
doctor3.health()
doctor3.train()
```

**Exercise 3:** Try hyperparameter tuning
```python
# After the basic run, tune the best model:
tuned = doctor.train(tune=True, n_iter=20)
print(f"Tuned score: {tuned.test_score:.4f}")
```

---
## What's Next?

| Notebook | Level | What you'll learn |
|----------|-------|-------------------|
| **You are here** | 🟢 Basic | fit → health → train → report |
| [Intermediate (15 min)](demo_intermediate.ipynb) | 🟡 Intermediate | Full 8-step pipeline + debugging + trust score |
| [Advanced (30 min)](demo_advanced.ipynb) | 🔴 Advanced | Tuning + custom models + auto-improve + feature impact |
| [Quick Start](quickstart_tabular.ipynb) | ⚡ Reference | All 8 steps in one clean notebook |

---
*KaizenStat v0.5.1 · [GitHub](https://github.com/masuddarrahaman/KaizenStat-Library) · MIT License*